In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

os.chdir('/content/drive/MyDrive/magang-ba-2026')
print("Direktori aktif sekarang berada di:", os.getcwd())

Direktori aktif sekarang berada di: /content/drive/MyDrive/magang-ba-2026


**Unduh model sesuai keputusan**

In [4]:
import os
import json

# 1. Pastikan direktori 'models' tersedia
os.makedirs("models", exist_ok=True)

# 2. Menyusun data Model Card sesuai panduan
model_card = {
    "model_id": "crypter70/IndoBERT-Sentiment-Analysis",
    "revision": "main",
    "license": "mit",
    "task": "text-classification",
    "label_mapping": {
        "positive": "positif",
        "neutral": "netral",
        "negative": "negatif"
    },
    "description": "Model IndoBERT fine-tuned untuk analisis sentimen bahasa Indonesia (Opsi B - Siap pakai)"
}

In [5]:
# 3. Menyimpan ke file models/T1_model_card.json
with open("models/T1_model_card.json", "w", encoding="utf-8") as f:
    json.dump(model_card, f, indent=4, ensure_ascii=False)

print("Berhasil membuat dan menyimpan models/T1_model_card.json")

Berhasil membuat dan menyimpan models/T1_model_card.json


**Ukur durasi inferensi 200 komentar validation**

In [14]:
import time
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import pipeline
import json

# 1. Pastikan data dimuat dan validation set (val_df) terdefinisi kembali
df_comments = pd.read_csv("data/processed/comments_v1.csv")
text_col = 'text_original'
label_col = 'sentiment_label'

df_clean = df_comments.dropna(subset=[text_col, label_col]).copy()
df_clean['reference_label'] = df_clean[label_col]

# Pembagian dataset secara stratified
train_val, test_df = train_test_split(
    df_clean, test_size=0.15, random_state=42, stratify=df_clean['reference_label']
)
train_df, val_df = train_test_split(
    train_val, test_size=0.1764, random_state=42, stratify=train_val['reference_label']
)

In [15]:
# 2. Menyiapkan subset 200 komentar dari data validation untuk pengukuran durasi inferensi
eval_subset = pd.concat([val_df] * 2, ignore_index=True).head(200)
texts_eval = eval_subset[text_col].tolist()

print(f"Menyiapkan {len(eval_subset)} komentar untuk uji inferensi...")

Menyiapkan 200 komentar untuk uji inferensi...


In [16]:
# 3. Memuat Model Card & Pipeline Transformer (Opsi B)
with open("models/T1_model_card.json", "r", encoding="utf-8") as f:
    model_card = json.load(f)

clf = pipeline(
    "text-classification",
    model=model_card["model_id"],
    revision=model_card["revision"],
    device=-1,          # Menggunakan CPU
    truncation=True,
    max_length=128
)

print(f"Menjalankan inferensi untuk {len(eval_subset)} komentar pada CPU...")

config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/709k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Menjalankan inferensi untuk 200 komentar pada CPU...


In [17]:
# 4. Mengukur durasi waktu inferensi secara nyata
start_time = time.time()
outputs = clf(texts_eval, batch_size=16)
end_time = time.time()

inference_duration = end_time - start_time
print(f"Durasi inferensi 200 komentar di CPU: {inference_duration:.4f} detik")

Durasi inferensi 200 komentar di CPU: 46.2371 detik


**Hitung metrik evaluasi pada validation set**

In [21]:
import json
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from transformers import pipeline

# 1. Pastikan data validation asli (val_df) termuat kembali dengan benar
df_comments = pd.read_csv("data/processed/comments_v1.csv")
text_col = 'text_original'
label_col = 'sentiment_label'

df_clean = df_comments.dropna(subset=[text_col, label_col]).copy()
df_clean['reference_label'] = df_clean[label_col]

train_val, test_df = train_test_split(
    df_clean, test_size=0.15, random_state=42, stratify=df_clean['reference_label']
)
train_df, val_df = train_test_split(
    train_val, test_size=0.1764, random_state=42, stratify=train_val['reference_label']
)

In [22]:
# 2. Muat model card & label_map
with open("models/T1_model_card.json", "r", encoding="utf-8") as f:
    model_card = json.load(f)

label_map = model_card["label_mapping"]

In [24]:
# 3. Jalankan inferensi langsung pada seluruh val_df (145 baris)
clf = pipeline(
    "text-classification",
    model=model_card["model_id"],
    revision=model_card["revision"],
    device=-1,
    truncation=True,
    max_length=128
)

texts_val = val_df[text_col].tolist()
val_outputs = clf(texts_val, batch_size=16)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [25]:
# 4. Hitung akurasi dan classification report
val_preds = [label_map.get(o["label"].lower(), o["label"]) for o in val_outputs]
val_acc = accuracy_score(val_df['reference_label'], val_preds)

val_report = classification_report(val_df['reference_label'], val_preds, output_dict=True, zero_division=0)

In [26]:
# 5. Menyimpan laporan validasi ke results/T1_val.json
t1_results = {
    "model_name": "T1_IndoBERT",
    "accuracy": val_acc,
    "inference_duration_200_comments_seconds": 46.2371, # Durasi dari Langkah 4
    "classification_report": val_report
}

os.makedirs("results", exist_ok=True)
with open("results/T1_val.json", "w", encoding="utf-8") as f:
    json.dump(t1_results, f, indent=4, ensure_ascii=False)

print("Berhasil menyimpan laporan validasi ke: results/T1_val.json")

Berhasil menyimpan laporan validasi ke: results/T1_val.json


In [27]:
# 6. Menambahkan ringkasan eksperimen ke results/experiments.csv
exp_data = {
    "experiment_id": ["T1_indobert"],
    "model_type": ["IndoBERT (Opsi B)"],
    "max_features": ["N/A (Tokenizer)"],
    "ngram_range": ["N/A"],
    "train_samples": [len(train_df)],
    "val_samples": [len(val_df)],
    "val_accuracy": [val_acc]
}
df_exp = pd.DataFrame(exp_data)

exp_path = "results/experiments.csv"
if os.path.exists(exp_path):
    df_exp.to_csv(exp_path, mode='a', header=False, index=False)
else:
    df_exp.to_csv(exp_path, index=False)

print("Berhasil menambahkan ringkasan eksperimen ke: results/experiments.csv")

Berhasil menambahkan ringkasan eksperimen ke: results/experiments.csv
